In [1]:
import gc
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# --- ПУТИ ---
BASE_DIR = Path(r"C:\Users\egor0\ProjectsVSCode\ML_Practice")
DATA_DIR = BASE_DIR / "data"
PARQUET_DIR = DATA_DIR / "parquet"
CLEAN_DIR = DATA_DIR / "clean"
CLEAN_DIR.mkdir(exist_ok=True)

# --- НАСТРОЙКИ ---
ENGINE = 'fastparquet'
RANDOM_STATE = 42

# --- НАСТРОЙКИ ОТОБРАЖЕНИЯ ---
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')


cat_cols_present = [
    'ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain',
    'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
    'DeviceType', 'DeviceInfo', 
    'id_12', 'id_15', 'id_16', 'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 
    'id_26', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 
    'id_36', 'id_37', 'id_38'
]


first_file = sorted(PARQUET_DIR.glob('train_trans_*.parquet'))[0]
df_sample = pd.read_parquet(first_file, engine=ENGINE)

all_cols = df_sample.columns.tolist()
service_cols = ['TransactionID', 'isFraud']


features_to_keep_final = [
    c for c in all_cols 
    if c not in service_cols and c not in cat_cols_present
]

del df_sample
gc.collect()

print(f"Папка с чистыми данными: {CLEAN_DIR}")
print(f"Найдено числовых признаков: {len(features_to_keep_final)}")
print(f"Найдено категориальных: {len(cat_cols_present)}")

Папка с чистыми данными: C:\Users\egor0\ProjectsVSCode\ML_Practice\data\clean
Найдено числовых признаков: 378
Найдено категориальных: 37


In [2]:
import gc
import numpy as np
import pandas as pd
from scipy.stats import spearmanr


df_train = pd.read_parquet(CLEAN_DIR / 'train_part.parquet', engine=ENGINE)
df_val = pd.read_parquet(CLEAN_DIR / 'val_part.parquet', engine=ENGINE)

X_train = df_train.drop(columns=['isFraud'])
y_train = df_train['isFraud']
del df_train
gc.collect()

# 1. Выделяем числовые и категориальные признаки
cat_cols_actual = [c for c in cat_cols_present if c in X_train.columns]
num_cols_actual = [c for c in X_train.columns if c not in cat_cols_actual]

print(f"Числовых: {len(num_cols_actual)}, Категориальных: {len(cat_cols_actual)}")

# 2. Сэмплируем данные для Спирмена
sample_size = min(50000, len(X_train))
sample_idx = np.random.choice(len(X_train), sample_size, replace=False)
X_sample = X_train[num_cols_actual].iloc[sample_idx].copy()
y_sample = y_train.iloc[sample_idx].copy()

# 3. Считаем корреляцию Спирмена с таргетом
target_corr = {}
for col in num_cols_actual:
    if X_sample[col].nunique() <= 1:
        target_corr[col] = 0
        continue
    corr, _ = spearmanr(X_sample[col], y_sample, nan_policy='omit')
    target_corr[col] = abs(corr) if not np.isnan(corr) else 0

target_corr_series = pd.Series(target_corr).sort_values(ascending=False)

weak_threshold = 0.01
features_to_keep = target_corr_series[target_corr_series > weak_threshold].index.tolist()
print(f"1. После отсечения слабых (< {weak_threshold}): {len(features_to_keep)} из {len(num_cols_actual)}")

corr_matrix = X_sample[features_to_keep].corr(method='spearman').abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = []
high_corr_threshold = 0.95

for column in upper.columns:
    high_corr_features = upper.index[upper[column] > high_corr_threshold].tolist()
    if len(high_corr_features) > 0:
        for hc_feature in high_corr_features:
            # Оставляем тот, у которого связь с таргетом выше
            if target_corr_series[column] < target_corr_series[hc_feature]:
                if column not in to_drop: to_drop.append(column)
            else:
                if hc_feature not in to_drop: to_drop.append(hc_feature)

features_to_keep_final = [f for f in features_to_keep if f not in to_drop]
print(f"2. После удаления линейно зависимых (> {high_corr_threshold}): {len(features_to_keep_final)} признаков")
print(f"Удалено из-за корреляции: {len(to_drop)}")

# 5. Обновляем списки признаков для дальнейшего использования
globals()['features_to_keep_final'] = features_to_keep_final
globals()['cat_cols_present'] = cat_cols_actual

# 6. Сохраняем отфильтрованные датасеты
final_cols = features_to_keep_final + cat_cols_actual + ['TransactionDT', 'isFraud']

final_cols = list(dict.fromkeys([c for c in final_cols if c in X_train.columns or c == 'isFraud']))

df_train_filtered = df_train_orig = X_train[final_cols[:-1]].copy()
df_train_filtered['isFraud'] = y_train
df_val_filtered = df_val[[c for c in final_cols if c in df_val.columns]].copy()

df_train_filtered.to_parquet(CLEAN_DIR / 'train_filtered.parquet', index=False, engine=ENGINE)
df_val_filtered.to_parquet(CLEAN_DIR / 'val_filtered.parquet', index=False, engine=ENGINE)

print(f"\nСохранено в train_filtered.parquet и val_filtered.parquet")
print(f"Итоговая размерность: {df_train_filtered.shape}")

del X_train, df_train_filtered, df_val_filtered, X_sample, corr_matrix
gc.collect()

Числовых: 378, Категориальных: 14
1. После отсечения слабых (< 0.01): 348 из 378
2. После удаления линейно зависимых (> 0.95): 212 признаков
Удалено из-за корреляции: 136

Сохранено в train_filtered.parquet и val_filtered.parquet
Итоговая размерность: (419221, 227)


0